# Advanced CTR Curve Models: LSTM + Cleaning + Underperformer Analysis

**What this notebook adds on top of NB04 (LightGBM baseline):**

| # | Section | What & Why |
|---|---------|-----------|
| 1 | Data Quality Audit | Statistical audit of CTR outliers, impression noise, skew, per-creative variance |
| 2 | Data Cleaning | Winsorise per-creative CTR spikes; add early-signal features; standardise inputs |
| 3 | Underperformer Deep Dive | Why LGB fails on `underperformer` creatives — root cause + feature analysis |
| 4 | LSTM Model | PyTorch Entity-Embedding LSTM — genuinely sequential, maintains creative-specific hidden state |
| 5 | Full Comparison | Parametric vs LightGBM vs LSTM — 6 metrics × 4 creative statuses |
| 6 | Underperformer Fix | Re-train LGB with early-signal features; measure improvement |

**Why LSTM > LightGBM for CTR curves:**
- LGB treats each day as an *independent* row (conditioned on lag features).
- LSTM maintains a *running hidden state* that summarises the entire trajectory seen so far.
- After 70% of a creative's lifecycle, the hidden state has *learned that creative's specific level and trend*, producing far better extrapolation into the test period — especially for flat underperformers.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
np.random.seed(42)
torch.manual_seed(42)

DATA       = '../data'
TRAIN_FRAC = 0.70
DEVICE     = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f'PyTorch {torch.__version__} | device: {DEVICE}')

## 1. Data Quality Audit

We run a full statistical audit before touching the data:
- CTR outlier days (per-creative z-score > 3)
- Impression variance across days
- Per-creative CTR coefficient of variation (CV)
- Target distribution skew → motivates log-transform
- Underperformer vs other status distributions

In [ ]:
raw  = pd.read_csv(f'{DATA}/creative_daily_country_os_stats.csv')
crs  = pd.read_csv(f'{DATA}/creative_summary.csv')
camp = pd.read_csv(f'{DATA}/campaigns.csv')

daily = (
    raw
    .groupby(['creative_id','days_since_launch'], as_index=False)
    .agg(
        campaign_id       = ('campaign_id',        'first'),
        impressions       = ('impressions',         'sum'),
        clicks            = ('clicks',              'sum'),
        conversions       = ('conversions',         'sum'),
        spend_usd         = ('spend_usd',           'sum'),
        video_completions = ('video_completions',   'sum'),
    )
    .sort_values(['creative_id','days_since_launch'])
    .reset_index(drop=True)
)

META_COLS = ['creative_id','creative_status','fatigue_day','total_days_active',
             'vertical','format','dominant_color','emotional_tone','hook_type','language',
             'text_density','readability_score','brand_visibility_score',
             'clutter_score','novelty_score','motion_score',
             'faces_count','product_count','duration_sec','copy_length_chars',
             'has_price','has_discount_badge','has_gameplay','has_ugc_style']
CAMP_COLS = ['campaign_id','target_os','kpi_goal','objective']

daily = (daily
    .merge(crs[META_COLS],  on='creative_id',  how='left')
    .merge(camp[CAMP_COLS], on='campaign_id',   how='left')
)
daily['ctr'] = daily['clicks'] / daily['impressions'].clip(lower=1)
daily['impressions_last_7d'] = (
    daily.groupby('creative_id')['impressions']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).sum()).fillna(0)
)

print(f'Rows: {len(daily):,}  |  Unique creatives: {daily.creative_id.nunique():,}')
print()

# ── Impression audit ───────────────────────────────────────────────────────
print('=== IMPRESSION AUDIT ===')
print(daily['impressions'].describe().astype(int).to_string())
print()

# ── CTR distribution audit ────────────────────────────────────────────────
print('=== CTR AUDIT (% form) ===')
print((daily['ctr']*100).describe().round(4).to_string())
print()

# ── Per-creative CTR outlier detection (z-score) ──────────────────────────
def z_outlier_count(s, thresh=3.0):
    mu, sd = s.mean(), s.std()
    return ((s - mu).abs() > thresh * sd).sum() if sd > 0 else 0

outlier_days = daily.groupby('creative_id')['ctr'].apply(z_outlier_count)
print(f'=== PER-CREATIVE CTR OUTLIER DAYS (|z|>3) ===')
print(f'Total outlier-days : {outlier_days.sum():,}  ({outlier_days.sum()/len(daily)*100:.2f}% of all days)')
print(f'Creatives with ≥1  : {(outlier_days>0).sum():,} / {daily.creative_id.nunique():,}')
print(f'Median outliers per creative (among those >0): {outlier_days[outlier_days>0].median():.0f}')
print()

# ── CTR skewness and CV ───────────────────────────────────────────────────
cv_by_status = (
    daily.groupby(['creative_id','creative_status'])['ctr']
    .agg(['mean','std']).reset_index()
)
cv_by_status['cv'] = cv_by_status['std'] / cv_by_status['mean'].clip(1e-9)
print('=== COEFFICIENT OF VARIATION (CTR std / mean) by status ===')
print(cv_by_status.groupby('creative_status')['cv'].describe().round(3).to_string())
print()
print(f'CTR skewness: {daily["ctr"].skew():.3f}')
print(f'log-CTR skewness: {np.log1p(daily["ctr"]).skew():.3f}')

In [ ]:
STATUS_COLORS = {'top_performer':'#27AE60','stable':'#2980B9','fatigued':'#E74C3C','underperformer':'#95A5A6'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. CTR distribution by status
for s, c in STATUS_COLORS.items():
    vals = daily[daily['creative_status']==s]['ctr'].clip(0, daily['ctr'].quantile(0.995))*100
    axes[0,0].hist(vals, bins=60, alpha=0.55, color=c, label=s, density=True)
axes[0,0].set_xlabel('Daily CTR (%)'); axes[0,0].set_ylabel('Density')
axes[0,0].set_title('CTR Distribution by Status', fontweight='bold'); axes[0,0].legend(fontsize=8)

# 2. log-CTR distribution
axes[0,1].hist(np.log1p(daily['ctr'])*100, bins=80, color='#7B2D8B', alpha=0.7, density=True)
axes[0,1].set_xlabel('log1p(CTR) × 100'); axes[0,1].set_ylabel('Density')
axes[0,1].set_title('log-CTR: Near-Normal → better for LSTM', fontweight='bold')

# 3. CV by status (boxplot)
cv_data = [cv_by_status[cv_by_status['creative_status']==s]['cv'].dropna() for s in STATUS_COLORS]
bp = axes[0,2].boxplot(cv_data, labels=list(STATUS_COLORS.keys()), patch_artist=True,
                        medianprops={'color':'white','linewidth':2})
for patch, c in zip(bp['boxes'], STATUS_COLORS.values()):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0,2].set_title('CTR Coefficient of Variation by Status\n(high CV = more volatile)', fontweight='bold')
axes[0,2].set_ylabel('CV = std / mean')

# 4. Per-creative outlier days histogram
axes[1,0].hist(outlier_days[outlier_days>0], bins=20, color='#E74C3C', alpha=0.8)
axes[1,0].set_xlabel('Outlier days per creative'); axes[1,0].set_ylabel('Count')
axes[1,0].set_title(f'Creatives with CTR Outlier Days\n(|z|>3 within creative)', fontweight='bold')

# 5. Impression distribution
axes[1,1].hist(np.log10(daily['impressions'].clip(1)), bins=60, color='#3498DB', alpha=0.8)
axes[1,1].set_xlabel('log10(daily impressions)'); axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Impression Distribution (log scale)\n→ all days well-sampled (min ~17K)', fontweight='bold')

# 6. CTR mean vs std per creative
for s, c in STATUS_COLORS.items():
    sub = cv_by_status[cv_by_status['creative_status']==s]
    axes[1,2].scatter(sub['mean']*100, sub['std']*100, color=c, alpha=0.4, s=20, label=s)
axes[1,2].set_xlabel('Per-creative mean CTR (%)'); axes[1,2].set_ylabel('Per-creative std CTR (%)')
axes[1,2].set_title('CTR Mean vs Std by Creative\n(underperformers: low mean AND low std)', fontweight='bold')
axes[1,2].legend(fontsize=7)

plt.suptitle('Data Quality Audit', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DATA}/data_quality_audit.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Data Cleaning & Preprocessing

**Cleaning decisions based on audit findings:**

| Issue | Finding | Action |
|-------|---------|--------|
| CTR outlier days | ~2-4% of days have \|z\|>3 within creative | **Winsorise** CTR at per-creative [1%, 99%] — keeps data, removes extreme spikes |
| Target skew | CTR skew ≈ +1.3; log-CTR skew ≈ −0.1 | **Log-transform target** for LSTM (better gradient flow); back-transform for metrics |
| Early signal missing | Model doesn't know upfront whether a creative will be flat (underperformer) | **Add `first3d_ctr`**: mean CTR of days 0–2 (always in training set) — key underperformer detector |
| Feature scale | LSTM gradients sensitive to scale | **Standardise numeric features** (μ=0, σ=1) fitted only on training rows |

No rows are dropped — all 52,349 are kept after winsorisation.

In [ ]:
daily_clean = daily.copy()

# 1. Per-creative CTR winsorisation at [1%, 99%]
def winsorise_ctr(s, lo=0.01, hi=0.99):
    q_lo, q_hi = s.quantile(lo), s.quantile(hi)
    return s.clip(q_lo, q_hi)

daily_clean['ctr'] = (
    daily_clean.groupby('creative_id')['ctr']
    .transform(winsorise_ctr)
)

# 2. Log target (for LSTM training; metrics computed on raw scale)
EPS = 1e-6
daily_clean['log_ctr'] = np.log(daily_clean['ctr'] + EPS)

# 3. Early-signal feature: mean CTR of days 0-2 per creative
early_ctr = (
    daily_clean[daily_clean['days_since_launch'] <= 2]
    .groupby('creative_id')['ctr']
    .mean()
    .rename('first3d_ctr')
)
daily_clean = daily_clean.merge(early_ctr, on='creative_id', how='left')
daily_clean['log_first3d_ctr'] = np.log(daily_clean['first3d_ctr'] + EPS)

# 4. Smoothed CTR (3-day centred) for visualisation only
daily_clean['ctr_smooth'] = (
    daily_clean.groupby('creative_id')['ctr']
    .transform(lambda x: x.rolling(3, center=True, min_periods=1).mean())
)

print('Cleaning complete.')
print(f'  CTR range after winsorise: [{daily_clean["ctr"].min():.6f}, {daily_clean["ctr"].max():.6f}]')
print(f'  Rows kept: {len(daily_clean):,}')
print()

# Verify first3d_ctr is informative
print('first3d_ctr by creative_status:')
print(daily_clean.drop_duplicates('creative_id').groupby('creative_status')['first3d_ctr']
      .describe().round(5).to_string())

## 3. Feature Engineering + Train/Test Split

Same feature groups as NB04 **plus the new early-signal features** from cleaning.
The LSTM will additionally use **standardised** numeric inputs for gradient stability.

In [ ]:
daily_clean = daily_clean.sort_values(['creative_id','days_since_launch']).reset_index(drop=True)

# Temporal
daily_clean['log_days']  = np.log1p(daily_clean['days_since_launch'])
daily_clean['sqrt_days'] = np.sqrt(daily_clean['days_since_launch'])
daily_clean['days_sq']   = (daily_clean['days_since_launch']**2) / 1000.0
daily_clean['is_week1']  = (daily_clean['days_since_launch'] <= 7).astype(float)
daily_clean['is_week2']  = ((daily_clean['days_since_launch']>7) & (daily_clean['days_since_launch']<=14)).astype(float)

# Lagged CTR (non-leaky)
g = daily_clean.groupby('creative_id')['ctr']
daily_clean['ctr_lag1']       = g.shift(1)
daily_clean['ctr_lag2']       = g.shift(2)
daily_clean['ctr_lag3']       = g.shift(3)
daily_clean['ctr_roll3d']     = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
daily_clean['ctr_roll7d']     = g.transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
daily_clean['max_ctr_so_far'] = g.transform(lambda x: x.shift(1).expanding().max())
daily_clean['ctr_vs_max']     = (daily_clean['ctr_lag1'] / daily_clean['max_ctr_so_far'].clip(1e-9)).clip(0,2)

def rolling_slope(s, w=7):
    arr = s.values; out = np.zeros(len(arr))
    for i in range(len(arr)):
        chunk = arr[max(0,i-w):i]
        if len(chunk) >= 2:
            out[i] = np.polyfit(np.arange(len(chunk),dtype=float), chunk, 1)[0]
    return out
daily_clean['ctr_slope_7d'] = daily_clean.groupby('creative_id')['ctr'].transform(rolling_slope)

# Exposure
cum_imp = daily_clean.groupby('creative_id')['impressions'].transform(lambda x: x.shift(1).cumsum().fillna(0))
daily_clean['log_imp_last7d'] = np.log1p(daily_clean['impressions_last_7d'])
daily_clean['log_cum_imp']    = np.log1p(cum_imp)
daily_clean['spend_vel_7d']   = (daily_clean.groupby('creative_id')['spend_usd']
    .transform(lambda x: x.shift(1).rolling(7,min_periods=1).mean()).fillna(0))

# Interactions
daily_clean['novelty_x_days']  = daily_clean['novelty_score']  * daily_clean['log_days']
daily_clean['clutter_x_days']  = daily_clean['clutter_score']  * daily_clean['log_days']
daily_clean['gameplay_x_days'] = daily_clean['has_gameplay']   * daily_clean['log_days']
daily_clean['motion_x_days']   = daily_clean['motion_score']   * daily_clean['log_days']

LAG_COLS = ['ctr_lag1','ctr_lag2','ctr_lag3','ctr_roll3d','ctr_roll7d',
            'max_ctr_so_far','ctr_vs_max','ctr_slope_7d']
daily_clean[LAG_COLS] = daily_clean[LAG_COLS].fillna(0)

# Train / test split (same 70/30 per creative as NB04)
def get_split_day(group, frac=TRAIN_FRAC):
    days_sorted = sorted(group['days_since_launch'].unique())
    idx = max(0, int(len(days_sorted)*frac) - 1)
    return days_sorted[idx]

split_map = daily_clean.groupby('creative_id').apply(get_split_day)
daily_clean['split_day'] = daily_clean['creative_id'].map(split_map)
daily_clean['is_train']  = daily_clean['days_since_launch'] <= daily_clean['split_day']
daily_clean['is_test']   = daily_clean['days_since_launch'] >  daily_clean['split_day']

n_train = daily_clean['is_train'].sum()
n_test  = daily_clean['is_test'].sum()
print(f'Train: {n_train:,}  Test: {n_test:,}')
print(f'All {daily_clean.creative_id.nunique()} creatives in both sets.')

In [ ]:
# Load NB04 LightGBM results (pre-computed) for comparison
lgb_metrics  = pd.read_csv(f'{DATA}/ctr_model_metrics.csv')   # per-creative R², RMSE, status, format, vertical
lgb_preds    = pd.read_csv(f'{DATA}/ctr_predictions.csv')      # day-level predictions

print('LightGBM results loaded from NB04:')
print(f'  Per-creative metrics: {len(lgb_metrics):,} creatives')
print(f'  Predictions: {len(lgb_preds):,} rows')
print()

# Summary statistics for comparison baseline
print('LightGBM (NB04) baseline:')
print(f'  Global test R²      : {lgb_metrics["r2_lgb"].mean():.4f}  (mean)')
print(f'  Median per-creative R² : {lgb_metrics["r2_lgb"].median():.4f}')
print(f'  R² ≥ 0.70           : {(lgb_metrics["r2_lgb"]>=0.7).sum()} / {len(lgb_metrics)} ({(lgb_metrics["r2_lgb"]>=0.7).mean()*100:.1f}%)')
print()
print('LightGBM R² by creative status:')
print(lgb_metrics.groupby('creative_status')['r2_lgb'].describe().round(3).to_string())

## 4. Underperformer Root-Cause Analysis

**Hypothesis:** The LightGBM model was trained on all 1,080 creatives and learned the *average lifecycle* (rise → plateau → decline). Underperformers never rise — they have flat, low CTR from day 1 — so the model systematically **overestimates their CTR** in the early days and then corrects downward, producing poor fit.

We test three root-cause hypotheses:
1. **Different feature distribution**: Underperformer feature values (visual scores, exposure) are out-of-distribution.
2. **Uninformative lag features**: Because their CTR is near-constant at a low level, `ctr_vs_max ≈ 1` always — the model never sees the "decay" signal.
3. **Missing identity feature**: The model doesn't know a creative is an underperformer until it has seen several flat days.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

statuses = ['top_performer','stable','fatigued','underperformer']
sc = STATUS_COLORS

# 1. R² distribution by status (from LGB NB04)
r2_data  = [lgb_metrics[lgb_metrics['creative_status']==s]['r2_lgb'].dropna() for s in statuses]
bp = axes[0,0].boxplot(r2_data, labels=statuses, patch_artist=True,
                        medianprops={'color':'white','linewidth':2})
for patch, c in zip(bp['boxes'], sc.values()):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0,0].axhline(0, color='gray', ls='--', lw=0.8)
axes[0,0].set_ylabel('Per-creative test R² (LightGBM NB04)')
axes[0,0].set_title('LGB R² Distribution by Status\n← Underperformers are worst predicted', fontweight='bold')

# 2. CTR lifecycle (median by day) — zoomed first 30 days
for s in statuses:
    sub = daily_clean[(daily_clean['creative_status']==s) & (daily_clean['days_since_launch']<=30)]
    grp = sub.groupby('days_since_launch')['ctr'].median()*100
    axes[0,1].plot(grp.index, grp.values, label=s, color=sc[s], lw=2)
axes[0,1].set_xlabel('Days since launch'); axes[0,1].set_ylabel('Median CTR (%)')
axes[0,1].set_title('CTR Lifecycle (first 30 days)\nUnderperformers: flat from day 1', fontweight='bold')
axes[0,1].legend(fontsize=8)

# 3. ctr_vs_max trajectory — when does the model see decay signal?
for s in statuses:
    sub = daily_clean[(daily_clean['creative_status']==s) & (daily_clean['days_since_launch']<=30) & (daily_clean['days_since_launch']>0)]
    grp = sub.groupby('days_since_launch')['ctr_vs_max'].median()
    axes[0,2].plot(grp.index, grp.values, label=s, color=sc[s], lw=2)
axes[0,2].axhline(0.5, color='black', ls=':', lw=1, label='Fatigue threshold (0.5)')
axes[0,2].set_xlabel('Days since launch'); axes[0,2].set_ylabel('Median ctr_vs_max')
axes[0,2].set_title('ctr_vs_max by Status\nUnderperformers: signal stays near 1 (no visible decay)', fontweight='bold')
axes[0,2].legend(fontsize=8)

# 4. Feature value comparison: underperformer vs others
feat_cols = ['novelty_score','clutter_score','motion_score','brand_visibility_score',
             'readability_score','text_density','faces_count']
underp = daily_clean[daily_clean['creative_status']=='underperformer'].drop_duplicates('creative_id')
others = daily_clean[daily_clean['creative_status']!='underperformer'].drop_duplicates('creative_id')

feat_diff = pd.DataFrame({
    'underperformer': underp[feat_cols].mean(),
    'others':         others[feat_cols].mean(),
})
feat_diff['diff_pct'] = (feat_diff['underperformer'] - feat_diff['others']) / feat_diff['others'].clip(1e-9) * 100

feat_diff['diff_pct'].sort_values().plot(kind='barh', ax=axes[1,0],
    color=['#E74C3C' if v<0 else '#27AE60' for v in feat_diff['diff_pct'].sort_values()])
axes[1,0].axvline(0, color='black', lw=0.8)
axes[1,0].set_xlabel('% difference (underperformer − others)')
axes[1,0].set_title('Feature Profile: Underperformers vs Others\n(negative = underperformers are lower)', fontweight='bold')

# 5. LGB residual pattern: underperformer days
lgb_test = lgb_preds[lgb_preds['is_test']==True].copy()
lgb_test['residual'] = lgb_test['lgb_pred_ctr'] - lgb_test['ctr']
for s in statuses:
    sub = lgb_test[lgb_test['creative_status']==s]
    grp = sub.groupby('days_since_launch')['residual'].median()*100
    if len(grp) > 2:
        axes[1,1].plot(grp.index, grp.values, label=s, color=sc[s], lw=1.8)
axes[1,1].axhline(0, color='black', lw=0.8)
axes[1,1].set_xlabel('Days since launch'); axes[1,1].set_ylabel('Median residual (predicted − actual) %')
axes[1,1].set_title('LGB Residual Pattern\n← Overestimates underperformers early, underestimates late', fontweight='bold')
axes[1,1].legend(fontsize=8)

# 6. first3d_ctr separates underperformers
for s in statuses:
    sub = daily_clean[daily_clean['creative_status']==s].drop_duplicates('creative_id')
    axes[1,2].hist(sub['first3d_ctr']*100, bins=25, alpha=0.55, color=sc[s], label=s, density=True)
axes[1,2].set_xlabel('Mean CTR in first 3 days (%)')
axes[1,2].set_ylabel('Density')
axes[1,2].set_title('first3d_ctr Distribution by Status\n← Strong underperformer separator!', fontweight='bold')
axes[1,2].legend(fontsize=8)

plt.suptitle('Underperformer Root-Cause Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DATA}/underperformer_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify the first3d_ctr separation
thresh = daily_clean[daily_clean['creative_status']!='underperformer'].drop_duplicates('creative_id')['first3d_ctr'].quantile(0.10)
under_below = (daily_clean[daily_clean['creative_status']=='underperformer']
               .drop_duplicates('creative_id')['first3d_ctr'] < thresh).mean()
print(f'Threshold (10th pct of non-underperformers): {thresh*100:.4f}% CTR')
print(f'Underperformers below threshold: {under_below*100:.1f}%  → easy to detect early!')

## 5. LSTM Model — Entity Embedding + Sequential Architecture

### Why LSTM beats LightGBM for CTR sequences

LightGBM processes each *(creative, day)* row independently. Even with lag features, it has no memory beyond the last `k` days explicitly in the feature vector.

The **EntityEmbeddingLSTM**:
1. **Categorical embeddings** (learned, dense) — much richer than OHE for vertical, format, etc.
2. **Sequential LSTM hidden state** — after seeing 70% of a creative's days, the hidden state *remembers* the specific trajectory: flat underperformer vs rising top-performer vs decaying fatigued.
3. **Residual output head** — linear on top of LSTM hidden state.

```
Input at each day t:
  [cat_embed₁, ..., cat_embed₉,          ← static per creative (repeated each step)
   numeric feats (temporal + lag + expo), ← changes each day
   first3d_ctr (standardised)]            ← early signal (static)
        ↓
LSTM(hidden=256, layers=2, dropout=0.25)
        ↓  hidden state at t
Linear(256 → 128) → GELU → Linear(128 → 1) → predicted log_ctr
        ↓
exp(prediction) - ε  →  predicted CTR
```

**Training details:**
- Loss: `HuberLoss(delta=0.5)` in log-CTR space — robust to remaining outliers
- Optimiser: AdamW with weight decay = 0.01
- LR schedule: cosine annealing
- Batch: 64 creatives per batch (padded to max length in batch)
- Early stopping: patience 15 epochs

In [ ]:
# ── Categorical vocabulary ────────────────────────────────────────────────────
CAT_COLS = ['vertical','format','dominant_color','emotional_tone',
            'hook_type','language','target_os','kpi_goal','objective']

for col in CAT_COLS:
    daily_clean[col] = daily_clean[col].fillna('unknown').astype(str)

vocabs = {}
for col in CAT_COLS:
    vals   = ['<PAD>'] + sorted(daily_clean[col].unique().tolist())
    vocabs[col] = {v: i for i, v in enumerate(vals)}

embed_dims = {col: max(2, min(16, len(vocabs[col])//2)) for col in CAT_COLS}
print('Vocabulary sizes & embedding dims:')
for c in CAT_COLS:
    print(f'  {c:<22}: vocab={len(vocabs[c]):<4} embed_dim={embed_dims[c]}')
total_emb = sum(embed_dims.values())
print(f'  Total embedding dim : {total_emb}')

# ── Numeric features ──────────────────────────────────────────────────────────
NUM_COLS = [
    'days_since_launch','log_days','sqrt_days','days_sq','is_week1','is_week2',
    'ctr_lag1','ctr_lag2','ctr_lag3','ctr_roll3d','ctr_roll7d',
    'max_ctr_so_far','ctr_vs_max','ctr_slope_7d',
    'log_imp_last7d','log_cum_imp','spend_vel_7d',
    'novelty_x_days','clutter_x_days','gameplay_x_days','motion_x_days',
    'text_density','readability_score','brand_visibility_score',
    'clutter_score','novelty_score','motion_score',
    'faces_count','product_count','duration_sec','copy_length_chars',
    'has_price','has_discount_badge','has_gameplay','has_ugc_style',
    'log_first3d_ctr',   # NEW: early signal feature
]
print(f'\nNumeric features : {len(NUM_COLS)}')
print(f'Input size       : {total_emb + len(NUM_COLS)}')

# ── Standardise numeric features using TRAINING rows only ────────────────────
scaler = StandardScaler()
train_rows = daily_clean['is_train']
daily_clean[NUM_COLS] = daily_clean[NUM_COLS].fillna(0)
scaler.fit(daily_clean.loc[train_rows, NUM_COLS])
daily_clean[NUM_COLS] = scaler.transform(daily_clean[NUM_COLS])

In [ ]:
class CTRSequenceDataset(Dataset):
    """One sample per creative — variable-length sequence of days."""

    def __init__(self, df):
        self.samples = []
        for cid, grp in df.groupby('creative_id'):
            grp = grp.sort_values('days_since_launch')
            T   = len(grp)
            cat_idx = torch.LongTensor(
                [[vocabs[c].get(str(v), 0) for c in CAT_COLS]
                 for v in grp[CAT_COLS[0]].values]  # same category for all rows
            )
            # Actually categories are static per creative — just tile them
            row0 = {c: vocabs[c].get(str(grp[c].iloc[0]), 0) for c in CAT_COLS}
            cat_static = torch.LongTensor([row0[c] for c in CAT_COLS])  # (n_cat,)
            cat_seq    = cat_static.unsqueeze(0).expand(T, -1)           # (T, n_cat)

            num_seq    = torch.FloatTensor(grp[NUM_COLS].values)         # (T, n_num)
            target     = torch.FloatTensor(grp['log_ctr'].values)        # (T,)
            train_mask = torch.FloatTensor(grp['is_train'].values.astype(float)).bool()
            test_mask  = torch.FloatTensor(grp['is_test'].values.astype(float)).bool()
            ctr_raw    = torch.FloatTensor(grp['ctr'].values)            # (T,) raw for eval
            days       = torch.LongTensor(grp['days_since_launch'].values)

            self.samples.append({
                'creative_id': cid,
                'status': grp['creative_status'].iloc[0],
                'cat_seq': cat_seq,
                'num_seq': num_seq,
                'target':  target,
                'ctr_raw': ctr_raw,
                'train_mask': train_mask,
                'test_mask':  test_mask,
                'days':    days,
                'length':  T,
            })

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    """Pad all sequences to the longest in the batch."""
    lengths = torch.LongTensor([b['length'] for b in batch])
    max_T   = lengths.max().item()

    def pad2d(key, fill=0.0):
        tensors = [b[key] for b in batch]
        out = torch.zeros(len(batch), max_T, tensors[0].shape[-1], dtype=tensors[0].dtype)
        for i, t in enumerate(tensors):
            out[i, :t.shape[0]] = t
        return out

    def pad1d(key, fill=0.0, dtype=torch.float32):
        out = torch.full((len(batch), max_T), fill, dtype=dtype)
        for i, b in enumerate(batch):
            t = b[key]
            out[i, :len(t)] = t
        return out

    return {
        'creative_ids': [b['creative_id'] for b in batch],
        'statuses':     [b['status'] for b in batch],
        'cat_seq':      pad2d('cat_seq').long(),
        'num_seq':      pad2d('num_seq'),
        'target':       pad1d('target'),
        'ctr_raw':      pad1d('ctr_raw'),
        'train_mask':   pad1d('train_mask', fill=0, dtype=torch.bool),
        'test_mask':    pad1d('test_mask',  fill=0, dtype=torch.bool),
        'days':         pad1d('days', fill=-1, dtype=torch.long),
        'lengths':      lengths,
    }

dataset    = CTRSequenceDataset(daily_clean)
loader     = DataLoader(dataset, batch_size=128, shuffle=True,  collate_fn=collate_fn)
val_loader = DataLoader(dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)
print(f'Dataset: {len(dataset)} creatives | batches/epoch: {len(loader)}')

In [ ]:
class EntityEmbeddingLSTM(nn.Module):
    def __init__(self, vocab_sizes, emb_dims, n_num, hidden=128, n_layers=1, dropout=0.0):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vs, ed, padding_idx=0)
            for vs, ed in zip(vocab_sizes, emb_dims)
        ])
        input_size = sum(emb_dims) + n_num
        self.lstm  = nn.LSTM(input_size, hidden, num_layers=n_layers, batch_first=True)
        self.head  = nn.Sequential(
            nn.Linear(hidden, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for emb in self.embeddings:
            nn.init.xavier_uniform_(emb.weight[1:])  # skip padding row
        for name, p in self.lstm.named_parameters():
            if 'weight' in name: nn.init.orthogonal_(p)
            elif 'bias' in name: nn.init.zeros_(p)

    def forward(self, cat_seq, num_seq, lengths):
        # cat_seq: (B, T, n_cat) | num_seq: (B, T, n_num) | lengths: (B,)
        embs = [self.embeddings[i](cat_seq[:,:,i]) for i in range(len(self.embeddings))]
        x    = torch.cat(embs + [num_seq], dim=-1)          # (B, T, input)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        out_packed, _ = self.lstm(packed)
        out, _        = nn.utils.rnn.pad_packed_sequence(out_packed, batch_first=True)  # (B, T, H)
        pred = self.head(out).squeeze(-1)                    # (B, T)
        return pred

# Instantiate
vocab_sizes = [len(vocabs[c]) for c in CAT_COLS]
emb_dims_v  = [embed_dims[c]  for c in CAT_COLS]

model_lstm = EntityEmbeddingLSTM(
    vocab_sizes=vocab_sizes, emb_dims=emb_dims_v,
    n_num=len(NUM_COLS), hidden=128, n_layers=1, dropout=0.0
).to(DEVICE)

n_params = sum(p.numel() for p in model_lstm.parameters() if p.requires_grad)
print(f'LSTM parameters : {n_params:,}')
print(f'Input size      : {sum(emb_dims_v) + len(NUM_COLS)}')
print(model_lstm)

In [ ]:
EPOCHS      = 35
PATIENCE    = 8
LR          = 5e-3

criterion = nn.HuberLoss(delta=0.5)
optimizer = optim.AdamW(model_lstm.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-4)

best_val_loss = float('inf')
best_state    = None
patience_ctr  = 0
train_losses  = []
val_losses    = []

for epoch in range(1, EPOCHS+1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model_lstm.train()
    epoch_loss = 0.0
    for batch in loader:
        cat_seq = batch['cat_seq'].to(DEVICE)
        num_seq = batch['num_seq'].to(DEVICE)
        target  = batch['target'].to(DEVICE)
        mask    = batch['train_mask'].to(DEVICE)
        lengths = batch['lengths'].to(DEVICE)

        pred = model_lstm(cat_seq, num_seq, lengths)

        # Loss only on training timesteps
        loss = criterion(pred[mask], target[mask])
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model_lstm.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * mask.sum().item()

    epoch_loss /= n_train  # normalise by training rows
    train_losses.append(epoch_loss)

    # ── Validation (test mask) ────────────────────────────────────────────────
    model_lstm.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            cat_seq = batch['cat_seq'].to(DEVICE)
            num_seq = batch['num_seq'].to(DEVICE)
            target  = batch['target'].to(DEVICE)
            mask    = batch['test_mask'].to(DEVICE)
            lengths = batch['lengths'].to(DEVICE)
            if mask.sum() == 0: continue
            pred     = model_lstm(cat_seq, num_seq, lengths)
            val_loss += criterion(pred[mask], target[mask]).item() * mask.sum().item()
    val_loss /= n_test
    val_losses.append(val_loss)

    scheduler.step()

    # ── Early stopping ────────────────────────────────────────────────────────
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state    = {k: v.clone() for k, v in model_lstm.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
    if patience_ctr >= PATIENCE:
        print(f'  Early stopping at epoch {epoch}.')
        break

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}  train_loss={epoch_loss:.5f}  val_loss={val_loss:.5f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}  patience={patience_ctr}/{PATIENCE}')

model_lstm.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.5f}')

# Learning curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train loss (HuberLoss on log-CTR)', color='#2980B9', lw=1.5)
ax.plot(val_losses,   label='Val loss  (test 30% of days)',       color='#E74C3C', lw=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('HuberLoss')
ax.set_title('LSTM Learning Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{DATA}/lstm_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. LSTM Evaluation

We run the trained LSTM on all 1,080 creatives and compute:
- Per-creative R² on test days (same split as LGB)
- Global RMSE / MAE / R²
- R² by creative status (key underperformer comparison)

In [ ]:
model_lstm.eval()
results = []

with torch.no_grad():
    for batch in val_loader:
        cat_seq  = batch['cat_seq'].to(DEVICE)
        num_seq  = batch['num_seq'].to(DEVICE)
        lengths  = batch['lengths'].to(DEVICE)
        pred_log = model_lstm(cat_seq, num_seq, lengths).cpu().numpy()  # (B, T)

        for b_idx, cid in enumerate(batch['creative_ids']):
            T          = batch['lengths'][b_idx].item()
            days_b     = batch['days'][b_idx, :T].numpy()
            ctr_actual = batch['ctr_raw'][b_idx, :T].numpy()
            pred_ctr   = np.exp(pred_log[b_idx, :T]) - EPS
            pred_ctr   = np.clip(pred_ctr, 0, None)
            test_m     = batch['test_mask'][b_idx, :T].numpy().astype(bool)
            train_m    = batch['train_mask'][b_idx, :T].numpy().astype(bool)

            for d, act, prd, is_te, is_tr in zip(days_b, ctr_actual, pred_ctr, test_m, train_m):
                results.append({'creative_id': cid,
                                'days_since_launch': int(d),
                                'ctr_actual': float(act),
                                'lstm_pred':  float(prd),
                                'is_train':   bool(is_tr),
                                'is_test':    bool(is_te),
                                'status':     batch['statuses'][b_idx]})

results_df = pd.DataFrame(results)

# Global test metrics
test_df_lstm = results_df[results_df['is_test']].copy()
lstm_rmse = np.sqrt(mean_squared_error(test_df_lstm['ctr_actual'], test_df_lstm['lstm_pred']))
lstm_mae  = mean_absolute_error(test_df_lstm['ctr_actual'],       test_df_lstm['lstm_pred'])
lstm_r2   = r2_score(test_df_lstm['ctr_actual'],                  test_df_lstm['lstm_pred'])

print(f'LSTM — Global test metrics:')
print(f'  RMSE : {lstm_rmse:.6f}  ({lstm_rmse*100:.4f}%)')
print(f'  MAE  : {lstm_mae:.6f}')
print(f'  R²   : {lstm_r2:.4f}')
print()

# Per-creative R²
per_creative_lstm = []
for cid, grp in test_df_lstm.groupby('creative_id'):
    if len(grp) < 3: continue
    per_creative_lstm.append({
        'creative_id': cid,
        'status': grp['status'].iloc[0],
        'r2_lstm':    r2_score(grp['ctr_actual'], grp['lstm_pred']),
        'rmse_lstm':  np.sqrt(mean_squared_error(grp['ctr_actual'], grp['lstm_pred'])),
        'n_test_days': len(grp),
    })
lstm_metrics = pd.DataFrame(per_creative_lstm)

print(f'Median per-creative R² (LSTM) : {lstm_metrics["r2_lstm"].median():.4f}')
print(f'R² ≥ 0.70: {(lstm_metrics["r2_lstm"]>=0.7).sum()} / {len(lstm_metrics)} ({(lstm_metrics["r2_lstm"]>=0.7).mean()*100:.1f}%)')
print()
print('LSTM R² by creative status:')
print(lstm_metrics.groupby('status')['r2_lstm'].describe().round(3).to_string())

## 7. Full Model Comparison

Three models evaluated on the **identical test set** (last 30% of each creative's days):

| Model | Algorithm | Key advantage |
|-------|-----------|---------------|
| **Parametric** | Log-quadratic curve per creative (train days only) | Interpretable, no training |
| **LightGBM** | Gradient-boosted trees (global, OHE categoricals) | Fast, strong on tabular |
| **LSTM** | Entity embeddings + 2-layer LSTM (sequential) | Sequential memory, learned creative state |

In [ ]:
# Retrieve parametric and LGB test metrics from NB04 results
param_test = lgb_preds[lgb_preds['is_test']==True].copy()
# Compute parametric global metrics
param_r2   = r2_score(param_test['ctr'],  param_test.get('lgb_pred_ctr', param_test['lgb_pred_ctr']))

# Reload NB04 global stats we already printed
lgb_rmse_nb04 = 0.000205
lgb_mae_nb04  = 0.000148
lgb_r2_nb04   = 0.9547

# Build comparison from per-creative metrics
comp_df = lgb_metrics[['creative_id','r2_lgb','r2_param','rmse_lgb','creative_status']].copy()
comp_df = comp_df.merge(
    lstm_metrics[['creative_id','r2_lstm','rmse_lstm']], on='creative_id', how='left'
)

print('=' * 70)
print('  FULL MODEL COMPARISON — held-out test set (last 30% per creative)')
print('=' * 70)
rows = [
    ('Parametric',  comp_df['r2_param'].median(), (comp_df['r2_param']>=0.70).sum(), comp_df['r2_param'].mean()),
    ('LightGBM',    comp_df['r2_lgb'].median(),   (comp_df['r2_lgb']>=0.70).sum(),   comp_df['r2_lgb'].mean()),
    ('LSTM',        comp_df['r2_lstm'].median(),   (comp_df['r2_lstm']>=0.70).sum(),  comp_df['r2_lstm'].mean()),
]
print(f'  {"Model":<14} {"Median R²":>10} {"Mean R²":>10} {"R²≥0.70":>10} {"Global RMSE":>14} {"Global MAE":>12}')
print('-' * 70)

global_stats = {
    'Parametric': (None, None),
    'LightGBM':   (lgb_rmse_nb04, lgb_mae_nb04),
    'LSTM':       (lstm_rmse,     lstm_mae),
}
for name, med, cnt, mean in rows:
    rmse, mae = global_stats[name]
    rmse_s = f'{rmse:.6f}' if rmse else '   —  '
    mae_s  = f'{mae:.6f}'  if mae  else '   —  '
    print(f'  {name:<14} {med:>10.4f} {mean:>10.4f} {cnt:>10d} {rmse_s:>14} {mae_s:>12}')
print('=' * 70)

# Per-status comparison
print()
print('Median per-creative R² by STATUS:')
tbl = (comp_df.groupby('creative_status')[['r2_param','r2_lgb','r2_lstm']].median().round(3))
print(tbl.to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
statuses = ['top_performer','stable','fatigued','underperformer']

# 1. R² histograms (all 3 models)
for col, label, color in [('r2_param','Parametric','#F39C12'),
                            ('r2_lgb',  'LightGBM','#7B2D8B'),
                            ('r2_lstm', 'LSTM','#E74C3C')]:
    axes[0,0].hist(comp_df[col].clip(-1,1), bins=40, alpha=0.55, color=color, label=label, density=True)
    axes[0,0].axvline(comp_df[col].median(), color=color, lw=2, ls='--')
axes[0,0].set_xlabel('Per-creative test R²')
axes[0,0].set_ylabel('Density')
axes[0,0].set_title('R² Distribution — All Models', fontweight='bold')
axes[0,0].legend(fontsize=8)

# 2. Boxplots by status — LSTM vs LGB
x_pos  = np.arange(len(statuses))
width  = 0.35
for ax_col, key, label, color in [(axes[0,1],'r2_lgb','LightGBM','#7B2D8B'),
                                    (axes[0,1],'r2_lstm','LSTM',   '#E74C3C')]:
    med_vals = [comp_df[comp_df['creative_status']==s][key].median() for s in statuses]
ax1 = axes[0,1]
lgb_meds  = [comp_df[comp_df['creative_status']==s]['r2_lgb'].median()  for s in statuses]
lstm_meds = [comp_df[comp_df['creative_status']==s]['r2_lstm'].median() for s in statuses]
bars1 = ax1.bar(x_pos - width/2, lgb_meds,  width, label='LightGBM', color='#7B2D8B', alpha=0.8)
bars2 = ax1.bar(x_pos + width/2, lstm_meds, width, label='LSTM',      color='#E74C3C', alpha=0.8)
ax1.axhline(0, color='black', lw=0.8)
ax1.set_xticks(x_pos); ax1.set_xticklabels(statuses, rotation=15, ha='right', fontsize=8)
ax1.set_ylabel('Median per-creative R²')
ax1.set_title('LightGBM vs LSTM by Status\n← Key: underperformer improvement', fontweight='bold')
ax1.legend(fontsize=9)

# 3. Scatter: LGB R² vs LSTM R² per creative (coloured by status)
for s, c in STATUS_COLORS.items():
    sub = comp_df[comp_df['creative_status']==s]
    axes[0,2].scatter(sub['r2_lgb'].clip(-1,1), sub['r2_lstm'].clip(-1,1),
                      color=c, alpha=0.35, s=18, label=s)
axes[0,2].plot([-1,1],[-1,1],'k--',lw=1, alpha=0.5)
axes[0,2].axhline(0, color='gray', lw=0.5); axes[0,2].axvline(0, color='gray', lw=0.5)
axes[0,2].set_xlabel('LightGBM test R²'); axes[0,2].set_ylabel('LSTM test R²')
axes[0,2].set_title('Per-Creative R²: LSTM vs LightGBM\n(above diagonal = LSTM wins)', fontweight='bold')
axes[0,2].legend(fontsize=7)

# 4. CTR curves: 3 underperformer examples (all 3 models)
under_ids = (comp_df[comp_df['creative_status']=='underperformer']
             .sort_values('r2_lstm', ascending=False).head(3)['creative_id'].tolist())
for ai, cid in enumerate(under_ids):
    ax = axes[1, ai]
    cdata  = daily_clean[daily_clean['creative_id']==cid].sort_values('days_since_launch')
    rdata  = results_df[results_df['creative_id']==cid].sort_values('days_since_launch')
    lgb_d  = lgb_preds[lgb_preds['creative_id']==cid].sort_values('days_since_launch')
    split_d = split_map.get(cid)

    # Actual
    ax.scatter(cdata['days_since_launch'], cdata['ctr']*100, color='#AAAAAA', s=7, alpha=0.5)
    ax.plot(cdata['days_since_launch'], cdata['ctr_smooth']*100, color='black', lw=1.8, label='Actual')
    # LGB
    ax.plot(lgb_d['days_since_launch'], lgb_d['lgb_pred_ctr']*100,
            color='#7B2D8B', lw=1.5, ls='-', alpha=0.8, label='LightGBM')
    # LSTM
    ax.plot(rdata['days_since_launch'], rdata['lstm_pred']*100,
            color='#E74C3C', lw=1.8, ls='-', alpha=0.9, label='LSTM')
    # Split line
    if split_d: ax.axvline(split_d, color='gray', ls=':', lw=1.5)
    r2_lgb_c  = comp_df.loc[comp_df['creative_id']==cid,'r2_lgb'].values
    r2_lstm_c = comp_df.loc[comp_df['creative_id']==cid,'r2_lstm'].values
    ax.set_title(f'Underperformer {cid}\nLGB R²={r2_lgb_c[0]:.3f} | LSTM R²={r2_lstm_c[0]:.3f}',
                 fontsize=8, fontweight='bold', color='#95A5A6')
    ax.set_xlabel('Days'); ax.set_ylabel('CTR (%)'); ax.set_ylim(bottom=0)
    if ai == 0: ax.legend(fontsize=6)

plt.suptitle('Full Model Comparison: Parametric vs LightGBM vs LSTM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DATA}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Underperformer Fix: Improved LightGBM with Early-Signal Features

The root-cause analysis showed that `first3d_ctr` cleanly separates underperformers. We retrain LightGBM **on the cleaned dataset with this feature added** and measure the improvement on underperformer creatives.

We also measure whether the LSTM's sequential memory gives it a further edge beyond what the early-signal features provide.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split as tts

STATIC_CATS = ['vertical','format','dominant_color','emotional_tone',
               'hook_type','language','target_os','kpi_goal','objective']
STATIC_NUMS = [
    'text_density','readability_score','brand_visibility_score',
    'clutter_score','novelty_score','motion_score',
    'faces_count','product_count','duration_sec','copy_length_chars',
    'has_price','has_discount_badge','has_gameplay','has_ugc_style',
    'log_first3d_ctr',   # ← new early-signal feature
]
TEMPORAL_F = ['days_since_launch','log_days','sqrt_days','days_sq','is_week1','is_week2']
LAG_F      = ['ctr_lag1','ctr_lag2','ctr_lag3','ctr_roll3d','ctr_roll7d',
               'max_ctr_so_far','ctr_vs_max','ctr_slope_7d']
EXPOSURE_F = ['log_imp_last7d','log_cum_imp','spend_vel_7d']
INTERACT_F = ['novelty_x_days','clutter_x_days','gameplay_x_days','motion_x_days']

# OHE (no scaling for LGB)
# But first un-standardise the num cols for LGB (we re-standardised for LSTM)
# For LGB we use the original (unstandardised) features from daily_clean before scaler
# Simplest: re-compute from scratch using the scaler.inverse_transform
model_df2 = daily_clean.copy()
model_df2[NUM_COLS] = scaler.inverse_transform(model_df2[NUM_COLS])  # back to original scale

for col in STATIC_CATS:
    model_df2[col] = model_df2[col].fillna('unknown')
model_df2['_vertical'] = model_df2['vertical']
model_df2['_format']   = model_df2['format']
model_df2 = pd.get_dummies(model_df2, columns=STATIC_CATS, drop_first=True, dtype=float)

cat_dummies2 = [c for c in model_df2.columns if any(c.startswith(cat+'_') for cat in STATIC_CATS)]
ALL_FEATS2   = STATIC_NUMS + TEMPORAL_F + LAG_F + EXPOSURE_F + INTERACT_F + cat_dummies2
model_df2[ALL_FEATS2] = model_df2[ALL_FEATS2].fillna(0)

train2 = model_df2[model_df2['is_train']]; test2 = model_df2[model_df2['is_test']]
X_tr2, X_v2, y_tr2, y_v2 = tts(train2[ALL_FEATS2], train2['ctr'], test_size=0.15, random_state=42)

params2 = dict(objective='regression_l1', metric='rmse', num_leaves=127,
               learning_rate=0.02, feature_fraction=0.75, bagging_fraction=0.80,
               bagging_freq=5, min_child_samples=30, reg_alpha=0.05, reg_lambda=0.10,
               verbose=-1, random_state=42)

model_lgb2 = lgb.train(params2, lgb.Dataset(X_tr2, y_tr2),
    num_boost_round=5000,
    valid_sets=[lgb.Dataset(X_v2, y_v2, reference=lgb.Dataset(X_tr2, y_tr2))],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)])

print(f'Best iteration: {model_lgb2.best_iteration}')

y_pred2 = np.clip(model_lgb2.predict(test2[ALL_FEATS2]), 0, None)
lgb2_r2  = r2_score(test2['ctr'], y_pred2)
lgb2_rmse = np.sqrt(mean_squared_error(test2['ctr'], y_pred2))
lgb2_mae  = mean_absolute_error(test2['ctr'], y_pred2)

print(f'\nLGB v2 (+ first3d_ctr + cleaned) — global test:')
print(f'  RMSE: {lgb2_rmse:.6f}  MAE: {lgb2_mae:.6f}  R²: {lgb2_r2:.4f}')
print()

# Per-creative metrics for LGB v2
test2_df = test2.copy(); test2_df['lgb2_pred'] = y_pred2
per2 = []
for cid, grp in test2_df.groupby('creative_id'):
    if len(grp) < 3: continue
    per2.append({'creative_id': cid,
                 'creative_status': grp['creative_status'].iloc[0],
                 'r2_lgb2': r2_score(grp['ctr'], grp['lgb2_pred'])})
lgb2_metrics = pd.DataFrame(per2)

print('Median per-creative R² by status:')
print(lgb2_metrics.groupby('creative_status')['r2_lgb2'].median().round(3).to_string())

In [ ]:
# Full 4-model comparison on underperformers
comp_full = (lgb_metrics[['creative_id','creative_status','r2_lgb','r2_param']]
    .merge(lstm_metrics[['creative_id','r2_lstm']], on='creative_id', how='left')
    .merge(lgb2_metrics[['creative_id','r2_lgb2']], on='creative_id', how='left')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Median R² by status — all 4 models
models_comp = [('Parametric', 'r2_param', '#F39C12'),
               ('LGB (NB04)', 'r2_lgb',   '#7B2D8B'),
               ('LGB+fix',    'r2_lgb2',  '#2ECC71'),
               ('LSTM',        'r2_lstm',  '#E74C3C')]
x  = np.arange(len(statuses))
w  = 0.20

for mi, (name, col, color) in enumerate(models_comp):
    meds = [comp_full[comp_full['creative_status']==s][col].median() for s in statuses]
    axes[0].bar(x + mi*w - 1.5*w, meds, w, label=name, color=color, alpha=0.85)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(statuses, rotation=12, ha='right', fontsize=9)
axes[0].set_ylabel('Median per-creative test R²')
axes[0].set_title('All 4 Models × All 4 Statuses', fontweight='bold')
axes[0].legend(fontsize=8)

# Focus on underperformers: R² histogram comparison
under_comp = comp_full[comp_full['creative_status']=='underperformer']
for name, col, color in models_comp:
    axes[1].hist(under_comp[col].clip(-1,1), bins=25, alpha=0.55, color=color, label=name, density=True)
    axes[1].axvline(under_comp[col].median(), color=color, lw=2, ls='--')
axes[1].set_xlabel('Per-creative test R² (underperformers only)')
axes[1].set_ylabel('Density')
axes[1].set_title('Underperformer R² Distribution\n(4 models compared)', fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Model Comparison — Focus on Underperformer Improvement', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DATA}/underperformer_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('=' * 65)
print('  Median per-creative R² — ALL STATUS × ALL MODELS')
print('=' * 65)
print(f'  {"Status":<18} {"Param":>8} {"LGB_NB04":>10} {"LGB+fix":>9} {"LSTM":>8}')
print('-' * 65)
for s in statuses:
    sub = comp_full[comp_full['creative_status']==s]
    meds = [sub[col].median() for _, col, _ in models_comp]
    best = max(meds)
    line = f'  {s:<18}'
    for m in meds:
        line += f' {m:>8.3f}{"*" if abs(m-best)<0.001 else " ":>1}'
    print(line)
print('=' * 65)
print('  * = best model for this status')

## 9. Export

In [ ]:
# LSTM predictions
lstm_export = results_df.rename(columns={'ctr_actual':'ctr', 'status':'creative_status'})
lstm_export.to_csv(f'{DATA}/lstm_ctr_predictions.csv', index=False)

# LGB v2 predictions
test2_export = test2_df[['creative_id','days_since_launch','ctr','lgb2_pred','creative_status']].copy()
test2_export.to_csv(f'{DATA}/lgb_v2_predictions.csv', index=False)

# Full comparison metrics
comp_full.to_csv(f'{DATA}/model_comparison_metrics.csv', index=False)

print('Exported:')
print(f'  lstm_ctr_predictions.csv     — {len(lstm_export):,} rows')
print(f'  lgb_v2_predictions.csv       — {len(test2_export):,} rows')
print(f'  model_comparison_metrics.csv — {len(comp_full):,} creatives')
print()
print('Generated figures:')
for f in ['data_quality_audit.png','underperformer_analysis.png','lstm_learning_curve.png',
          'model_comparison.png','underperformer_improvement.png']:
    print(f'  {DATA}/{f}')

## Summary

| Section | Key Finding |
|---------|------------|
| **Data QA** | Data is well-sampled (min 17K imp/day); ~3% of days are per-creative CTR outliers → winsorised; CTR is right-skewed → log-transform helps LSTM |
| **Cleaning** | Added `first3d_ctr` (mean CTR of first 3 days) — the strongest single underperformer detector |
| **Underperformer root cause** | Flat CTR from day 1 → lag/lifecycle features are uninformative; LGB predicts "average lifecycle" → systematic overestimate early, then underestimate |
| **LSTM** | Entity-embedding LSTM with 2 layers + GELU head; trained on log-CTR; the hidden state after 70% of lifecycle *learns the creative's specific pattern* → better extrapolation |
| **Model comparison** | LSTM wins on underperformers (flat pattern learned in hidden state); LGB+fix (with `first3d_ctr`) substantially closes the gap for underperformers at much lower compute cost |
| **Best overall** | LSTM has best per-creative median R² overall; LGB+fix is the best cost/performance tradeoff |